# Batching Benchmark

This notebook measures throughput at different batch sizes.

Goals:
- see why batching often improves throughput
- compare latency per example
- connect the result to deployment choices


In [ ]:
import time

import pandas as pd
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
)
model.eval()

batch_sizes = [1, 8, 32, 128, 512]
rows = []

for batch_size in batch_sizes:
    x = torch.randn(batch_size, 256)

    for _ in range(20):
        with torch.no_grad():
            model(x)

    start = time.perf_counter()
    runs = 200
    for _ in range(runs):
        with torch.no_grad():
            model(x)
    avg_ms = (time.perf_counter() - start) * 1000 / runs

    rows.append(
        {
            "batch_size": batch_size,
            "avg_batch_ms": avg_ms,
            "avg_per_example_ms": avg_ms / batch_size,
            "examples_per_second": batch_size / (avg_ms / 1000),
        }
    )

results = pd.DataFrame(rows)
results


In [ ]:
results.plot(
    x="batch_size",
    y=["avg_per_example_ms", "examples_per_second"],
    subplots=True,
    figsize=(8, 6),
    grid=True,
    title=["Latency per example", "Throughput"],
)
